# Nhiệm vụ 7: Đánh giá Chất lượng Cụm (Cluster Quality Evaluation)

**Mục đích:** Đánh giá chất lượng của mô hình Ward thông qua 5 chỉ số nội bộ (internal metrics) trải dài trên 15 snapshot (11/2023 - 01/2025). Từ đó, xác định bằng chứng toán học để hiểu đặc tính của các cụm được tạo ra ở cấu hình K=2 (đã được khóa).

In [ ]:
import pandas as pd
import json
import os
import math
from statistics import mean, median
from collections import Counter, defaultdict
import matplotlib.pyplot as plt

ward_dir = '../artifacts/m2-task5-ward-v1'
eval_dir = '../artifacts/m2-evaluation-ward'
os.makedirs(eval_dir, exist_ok=True)

# 1. Đọc manifest để lấy run_id và cấu hình
with open(os.path.join(ward_dir, 'manifest.json'), 'r', encoding='utf-8') as f:
    manifest = json.load(f)
run_id = manifest['run_id']
config = manifest['config']
k_range = config['clustering']['k_range']

# 2. Đọc Assignments và Diagnostics
assignments = []
with open(os.path.join(ward_dir, 'assignments.jsonl'), 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip(): assignments.append(json.loads(line))

diagnostics = []
with open(os.path.join(ward_dir, 'diagnostics.jsonl'), 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip(): diagnostics.append(json.loads(line))

by_date = defaultdict(dict)
for row in assignments:
    by_date[row['snapshot_date']][row['security_id']] = row
days = sorted(by_date.keys())

diag_index = {(d['snapshot_date'], d['k']): d for d in diagnostics}

QUALITY_METRICS = ["silhouette", "davies_bouldin", "calinski_harabasz", "inertia", "cluster_balance"]
quality_rows = []

# 3. Tính toán và Trích xuất các chỉ số chất lượng
for day in days:
    members = by_date[day]
    sizes = Counter(row["raw_cluster_id"] for row in members.values())
    
    for k_val in k_range:
        diag = diag_index.get((day, k_val))
        if not diag: continue
        
        row_data = {
            "method": "ward",
            "run_id": run_id,
            "snapshot_date": day,
            "k": k_val,
            "n_eligible": len(members),
            "smallest_cluster": min(sizes.values()) if sizes else 0,
            "largest_cluster": max(sizes.values()) if sizes else 0,
            "status": "ok" if diag.get("converged") else "unconverged"
        }
        for metric in QUALITY_METRICS:
            row_data[metric] = diag.get(metric)
            
        # Recalculate cluster_balance accurately for K=2 if needed
        if k_val == 2 and sizes:
            row_data["cluster_balance"] = min(sizes.values()) / max(sizes.values())
            
        quality_rows.append(row_data)

df_quality_all = pd.DataFrame(quality_rows)
df_quality_all.to_csv(os.path.join(eval_dir, 'quality_metrics_comparison.csv'), index=False)

# Lọc riêng K=2 để đánh giá xuyên suốt 15 tháng
df_quality = df_quality_all[df_quality_all['k'] == 2].copy()

# 4. Hàm Summary
def summarize(rows, metrics):
    result = []
    for metric in metrics:
        values = [r[metric] for r in rows if r.get(metric) is not None and not pd.isna(r[metric])]
        result.append({
            "metric": metric, "n_total": len(rows), "n_available": len(values),
            "mean": mean(values) if values else None,
            "median": median(values) if values else None,
            "minimum": min(values) if values else None,
            "maximum": max(values) if values else None
        })
    return result

summary_rows = summarize(df_quality.to_dict('records'), QUALITY_METRICS)
df_summary = pd.DataFrame(summary_rows)
df_summary.to_csv(os.path.join(eval_dir, 'quality_summary.csv'), index=False)

print("Đã sinh ra các file evaluation cho Task 7:\n- quality_metrics_comparison.csv\n- quality_summary.csv\n")
print("BẢNG TỔNG HỢP CHỈ SỐ TRUNG BÌNH (K=2):")
display(df_summary)

## Trực quan hóa và Đánh giá chi tiết (K=2 theo thời gian)
Do số lượng cụm (K) đã được khóa ở K=2 từ Nhiệm vụ 3, ta sẽ đánh giá sự ổn định và chất lượng của các chỉ số này qua 15 snapshots (từ tháng 11/2023 đến tháng 01/2025). 
Điều này tận dụng triệt để dữ liệu 15 tháng từ file `quality_metrics_comparison.csv` thay vì chỉ tính trung bình lại.

### Phần 7.1 — Silhouette Score

**Mục đích:**
Đánh giá mức độ các điểm gần cụm của mình và xa cụm khác. Cao hơn thường tốt hơn.

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(df_quality["snapshot_date"], df_quality["silhouette"], marker="o", color="#1f77b4", linewidth=2)
plt.title("Silhouette Score qua các tháng (Max is Better)")
plt.xticks(rotation=45)
plt.grid(True, linestyle="--", alpha=0.6)
plt.show()

**Đánh giá Silhouette Score:**
- Điểm trung bình (0.763) và trung vị (0.755) nằm ở mức rất cao, cho thấy K=2 phân tách thị trường thành hai nhóm rất rõ rệt.
- Tuy nhiên, có sự biến động lớn. Vào tháng 7/2024, điểm rớt xuống 0.458. Từ tháng 8/2024 trở đi (khi số mã tăng vọt lên ~600), Silhouette lại đạt mức đỉnh >0.9.
- Điều này ngụ ý rằng cấu trúc thị trường 2 cụm rất rõ nét ở quy mô thị trường lớn, nhưng có thể bị ảnh hưởng bởi những giai đoạn tái cấu trúc hoặc biến động dữ liệu mạnh.

### Phần 7.2 — Davies-Bouldin

**Mục đích:**
Đánh giá độ gọn và mức tách biệt của các cụm. Thấp hơn thường tốt hơn.

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(df_quality["snapshot_date"], df_quality["davies_bouldin"], marker="o", color="#ff7f0e", linewidth=2)
plt.title("Davies-Bouldin qua các tháng (Min is Better)")
plt.xticks(rotation=45)
plt.grid(True, linestyle="--", alpha=0.6)
plt.show()

**Đánh giá Davies-Bouldin:**
- Trung vị ở mức 0.534, khá thấp (tốt). 
- Các giai đoạn thị trường tăng quy mô (cuối 2024 - đầu 2025), DB càng giảm mạnh (0.26 - 0.33), xác nhận 2 cụm này cực kỳ đặc và cách xa nhau.
- Các gai nhọn (spike) ở tháng 03/2024 và 07/2024 tương đồng nghịch với Silhouette, cho thấy đây là những tháng các cụm bị nhiễu và kém gọn nhất.

### Phần 7.3 — Calinski-Harabasz

**Mục đích:**
So sánh phân tán giữa cụm và trong cụm. Cao hơn thường tốt hơn.

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(df_quality["snapshot_date"], df_quality["calinski_harabasz"], marker="o", color="#2ca02c", linewidth=2)
plt.title("Calinski-Harabasz qua các tháng (Max is Better)")
plt.xticks(rotation=45)
plt.grid(True, linestyle="--", alpha=0.6)
plt.show()

**Đánh giá Calinski-Harabasz:**
- Điểm dao động rất mạnh, dao động từ 118 đến 1882. 
- Mức tăng đột biến từ tháng 9/2024 trở đi phù hợp với số lượng mã tăng đột biến (từ ~240 lên ~600). Do công thức CH có phạt bởi $N$, khi $N$ tăng và cụm rõ rệt thì CH tăng cực lớn. 
- Nhìn chung, kết quả nhất quán ủng hộ cấu trúc 2 cụm tách rời của Ward.

### Phần 7.4 — Inertia

**Mục đích:**
Đo khoảng cách các điểm tới tâm cụm; chủ yếu dùng với K-Means.

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(df_quality["snapshot_date"], df_quality["inertia"], marker="o", color="#d62728", linewidth=2)
plt.title("Inertia qua các tháng (Depends on N)")
plt.xticks(rotation=45)
plt.grid(True, linestyle="--", alpha=0.6)
plt.show()

**Đánh giá Inertia:**
- Inertia không chuẩn hóa theo số lượng quan sát. Do đó, đỉnh cao ở tháng 11/2024 (419,136) là hệ quả trực tiếp của việc có 780 mã tham gia phân cụm.
- Trừ khi so sánh các K khác nhau trong CÙNG MỘT tháng, giá trị Inertia tuyệt đối không mang nhiều ý nghĩa để đánh giá chất lượng phân cụm theo chuỗi thời gian. 

### Phần 7.5 — Cluster balance

**Mục đích:**
Kiểm tra có cụm quá lớn hoặc quá nhỏ không.

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(df_quality["snapshot_date"], df_quality["cluster_balance"], marker="o", color="#9467bd", linewidth=2)
plt.title("Cluster Balance qua các tháng (Avoid Fragmentation)")
plt.xticks(rotation=45)
plt.axhline(y=0.05, color="r", linestyle=":", label="5% Threshold")
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend()
plt.show()

**Đánh giá Cluster Balance:**
- Trung vị Balance chỉ ở mức 0.067 (~6.7%), và giai đoạn cuối năm 2024 giảm xuống dưới ngưỡng 2% (vd: 12/2024 chỉ có 11 mã ở cụm nhỏ so với 578 mã cụm lớn).
- Điều này cảnh báo về tính "phân mảnh" (fragmentation). Mặc dù Silhouette và DB rất tốt, đó có thể là do Ward tách ra một nhóm nhỏ các cổ phiếu dị biệt (outliers) đứng riêng thành một cụm, trong khi phần lớn thị trường bị dồn vào cụm còn lại.
- Đây là đặc tính đặc trưng của Ward khi dữ liệu có đuôi dài (heavy tails) mà RobustScaler không xử lý hết (do clipping bị tắt). Do đó, cấu trúc K=2 tuy đẹp về toán học nhưng mang tính phân cực (1 nhóm khổng lồ vs 1 nhóm siêu nhỏ).

## Kết luận

- Phương pháp Ward tại K=2 mang lại các điểm số toán học cực kỳ ấn tượng (Silhouette >0.75, DB <0.6), chứng tỏ có ranh giới rõ rệt giữa hai nhóm.
- Tuy nhiên, sự phân hóa này chủ yếu đến từ việc chia thị trường thành một nhóm **rất lớn** và một nhóm **rất nhỏ** (Balance cực thấp).
- Cần thận trọng khi sử dụng kết quả này cho mục đích Portfolio, vì cụm thiểu số có thể chỉ là các mã có hành vi cực đoan chứ không đại diện cho một phân khúc thị trường truyền thống.